# 03. PD Logistic Regression

The target models probability of good standing (`good_bad = 1`). Default probability is calculated in the next notebook as `1 - P(good)`.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from scipy.stats import norm

DATA_PATH = Path('../data/loan_data_2007_2014.csv')
data = pd.read_csv(DATA_PATH, low_memory=False)
bad_statuses = {'Charged Off', 'Default', 'Does not meet the credit policy. Status:Charged Off', 'Late (31-120 days)'}
data['good_bad'] = (~data['loan_status'].isin(bad_statuses)).astype(int)
data['annual_inc'] = data['annual_inc'].fillna(data['annual_inc'].median())
data['dti'] = data['dti'].fillna(data['dti'].median())
model_data = pd.get_dummies(data[['grade', 'home_ownership', 'verification_status', 'purpose', 'term', 'int_rate', 'annual_inc', 'dti', 'good_bad']], columns=['grade', 'home_ownership', 'verification_status', 'purpose', 'term'], dtype=int)
model_data = model_data.replace([np.inf, -np.inf], np.nan).dropna()
X = model_data.drop(columns=['good_bad', 'grade_G'], errors='ignore')
y = model_data['good_bad']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## Broader and refined logistic models

The broader model includes all prepared predictors. The refined model keeps the stronger, interpretable families; both estimate probability of good standing.

In [ ]:
broad_model = LogisticRegression(max_iter=1000, solver='liblinear')
broad_model.fit(X_train, y_train)
broad_auc = roc_auc_score(y_test, broad_model.predict_proba(X_test)[:, 1])

refined_features = [column for column in X.columns if column.startswith(('grade_', 'term_', 'verification_status_', 'int_rate', 'dti'))]
refined_model = LogisticRegression(max_iter=1000, solver='liblinear')
refined_model.fit(X_train[refined_features], y_train)
refined_prob_good = refined_model.predict_proba(X_test[refined_features])[:, 1]
refined_auc = roc_auc_score(y_test, refined_prob_good)
pd.DataFrame({'model': ['broader', 'refined'], 'AUC': [broad_auc, refined_auc], 'Gini': [2 * broad_auc - 1, 2 * refined_auc - 1]})

In [ ]:
coefficients = pd.DataFrame({'feature': refined_features, 'coefficient': refined_model.coef_[0]})
coefficients['odds_ratio'] = np.exp(coefficients['coefficient'])
coefficients.sort_values('coefficient').head(20)

In [ ]:
# Approximate Wald p-values are shown for educational interpretation, not formal model approval.
design = np.c_[np.ones(len(X_train)), X_train[refined_features].to_numpy()]
probability = refined_model.predict_proba(X_train[refined_features])[:, 1]
covariance = np.linalg.pinv(design.T @ (design * (probability * (1 - probability))[:, None]))
standard_error = np.sqrt(np.diag(covariance))[1:]
coefficients['p_value'] = 2 * norm.sf(np.abs(coefficients['coefficient'] / standard_error))
coefficients.sort_values('p_value').head(20)

## Next stage

Notebook 04 converts `P(good)` to PD, reports AUC/Gini and a confusion matrix, then maps the refined model to an illustrative 300–850 scorecard.